In [18]:
import kagglehub
import pandas as pd
import os
import spacy
import json
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm
from collections import Counter

In [2]:
dataset_path = kagglehub.dataset_download("davidgauthier/glassdoor-job-reviews")

print("Path to dataset files:", dataset_path)

Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\davidgauthier\glassdoor-job-reviews\versions\11


In [3]:
csv_file = [f for f in os.listdir(dataset_path) if f.endswith('.csv')][0]
full_path = os.path.join(dataset_path, csv_file)

df = pd.read_csv(full_path)

df.head()

,firm,date_review,job_title,current,location,overall_rating,work_life_balance,culture_values,diversity_inclusion,career_opp,comp_benefits,senior_mgmt,recommend,ceo_approv,outlook,headline,pros,cons
0,AFH-Wealth-Management,2015-04-05,,Current Employee,NaN,2,4.0,3.0,NaN,2.0,3.0,3.0,x,o,r,"Young colleagues, poor micro management",Very friendly and welcoming to new staff. Easy...,"Poor salaries, poor training and communication."
1,AFH-Wealth-Management,2015-12-11,Office Administrator,"Current Employee, more than 1 year","Bromsgrove, England, England",2,3.0,1.0,NaN,2.0,1.0,4.0,x,o,r,"Excellent staff, poor salary","Friendly, helpful and hard-working colleagues",Poor salary which doesn't improve much with pr...
2,AFH-Wealth-Management,2016-01-28,Office Administrator,"Current Employee, less than 1 year","Bromsgrove, England, England",1,1.0,1.0,NaN,1.0,1.0,1.0,x,o,x,"Low salary, bad micromanagement",Easy to get the job even without experience in...,"Very low salary, poor working conditions, very..."
3,AFH-Wealth-Management,2016-04-16,,Current Employee,NaN,5,2.0,3.0,NaN,2.0,2.0,3.0,x,o,r,Over promised under delivered,Nice staff to work with,No career progression and salary is poor
4,AFH-Wealth-Management,2016-04-23,Office Administrator,"Current Employee, more than 1 year","Bromsgrove, England, England",1,2.0,1.0,NaN,2.0,1.0,1.0,x,o,x,client reporting admin,"Easy to get the job, Nice colleagues.","Abysmal pay, around minimum wage. No actual tr..."


In [4]:
rename_map = {
    "firm": "company_name",
    "date_review": "date_review",
    "ceo_approv": "ceo_approved_review",
    "career_opp": "career_opportunities",
    "comp_benefits": "compensation_and_benefits",
    "culture_values": "culture_and_values",
    "senior_mgmt": "senior_management",
    "work_life": "work_life_balance",
    "recommend": "recommend_to_a_friend",
    "outlook": "overall_outlook"
}

df = df.rename(columns=lambda c: rename_map.get(c, c.replace("_", " ")))

for col in df.columns:
    print(col)

company_name
date_review
job title
current
location
overall rating
work life balance
culture_and_values
diversity inclusion
career_opportunities
compensation_and_benefits
senior_management
recommend_to_a_friend
ceo_approved_review
overall_outlook
headline
pros
cons


In [5]:
# Remove duplicates
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_after = len(df)
n_removed = n_before - n_after
print(f"Removed {n_removed} duplicate rows.")

Removed 33515 duplicate rows.


In [60]:
# Remove rows with broken "KEY NOT FOUND" values (do not remap them)
bad_values = [
    "KEY NOT FOUND: jobLine.per_diem-former",
    "KEY NOT FOUND: jobLine.temporary-former",
    "Former Temporary Employee"
]

bad_mask = df["current"].isin(bad_values)
removed_n = int(bad_mask.sum())

df = df.loc[~bad_mask].copy()
print(f"Removed rows with KEY NOT FOUND: {removed_n}")

Removed rows with KEY NOT FOUND: 4


In [61]:
for c in df["current"].dropna().unique():
    print(c)

Current Employee
Current Employee, more than 1 year
Current Employee, less than 1 year
Former Employee
Current Employee, more than 5 years
Former Employee, more than 1 year
Former Employee, more than 3 years
Former Employee, more than 5 years
Current Employee, more than 3 years
Current Employee, more than 8 years
Former Employee, less than 1 year
Former Employee, more than 8 years
Current Employee, more than 10 years
Former Employee, more than 10 years
Former Contractor, less than 1 year
Former Intern, less than 1 year
Current Contractor, less than 1 year
Former Contractor
Former Intern, more than 1 year
Current Contractor
Former Intern
Current Intern, less than 1 year
Current Contractor, more than 1 year
Former Contractor, more than 1 year
Former Contractor, more than 8 years
Current Freelancer, more than 3 years


In [62]:
df["status"] = df["current"].str.extract(r"^(Current|Former)\b", expand=False)
df["employment type"] = df["current"].str.extract(r"^(?:Current|Former)\s+([^,]+)", expand=False)
df["seniority"] = df["current"].str.extract(r"^(?:Current|Former)\s+[^,]+(?:,\s*(.*))?$", expand=False)

display(df[["current", "status", "employment type", "seniority"]].head(10))
display(df["status"].value_counts(dropna=False))

,current,status,employment type,seniority
0,Current Employee,Current,Employee,None
1,"Current Employee, more than 1 year",Current,Employee,more than 1 year
2,"Current Employee, less than 1 year",Current,Employee,less than 1 year
3,Current Employee,Current,Employee,None
4,"Current Employee, more than 1 year",Current,Employee,more than 1 year
5,"Current Employee, less than 1 year",Current,Employee,less than 1 year
6,Former Employee,Former,Employee,None
7,"Current Employee, more than 5 years",Current,Employee,more than 5 years
8,"Former Employee, more than 1 year",Former,Employee,more than 1 year
9,"Former Employee, more than 3 years",Former,Employee,more than 3 years


status
Current    471188
Former     333859
Name: count, dtype: int64

In [63]:
df = df.drop(columns=['current'])

In [64]:
# Rename 'status' column to 'is_current_employee' and map values to True/False
df = df.rename(columns={'status': 'is_current_employee'})
df['is_current_employee'] = df['is_current_employee'].map({'Current': True, 'Former': False})

In [65]:
# Map the values in the specified columns
maps = {"v": "yes", "r": "neutral", "x": "no"}
cols = ["ceo_approved_review", "overall_outlook", "recommend_to_a_friend"]
for col in cols:
    df[col] = df[col].map(maps).astype("category")

df[cols].head()

,ceo_approved_review,overall_outlook,recommend_to_a_friend
0,NaN,neutral,no
1,NaN,neutral,no
2,NaN,no,no
3,NaN,neutral,no
4,NaN,no,no


In [66]:
df["cons_review_length"] = df["cons"].astype(str).str.split().apply(len)
df["pros_review_length"] = df["pros"].astype(str).str.split().apply(len)

In [67]:
df.to_csv("glassdoor_job_reviews.csv", index=False)

In [ ]:
df = df.head(10000)

In [48]:
def optimize_with_pipe(texts):
    processed_list = []

    # Add custom domain-specific stopwords
    domain_stops = {'company', 'place', 'get', 'lot', 'stay', 'go', 'good'}

    # 1. Load the model while disabling everything not needed for lemmatization
    # This saves a lot of RAM and CPU resources
    nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])
    
    for doc in tqdm(nlp.pipe(texts.astype(str), batch_size=4000), total=len(texts)):
        tokens = [
            token.lemma_.lower() for token in doc 
            if (
                not token.is_stop
                and token.is_alpha 
                and len(token.text) > 1 
                and token.lemma_.lower() not in domain_stops
                and token.pos_ in ['NOUN', 'PROPN', 'ADJ']
            )
        ]
        
        processed_list.append(" ".join(tokens))
        
    return processed_list

# 2. Execute
print("Processing 'Pros'...")
df['pros_optimized'] = optimize_with_pipe(df['pros'])

print("Processing 'Cons'...")
df['cons_optimized'] = optimize_with_pipe(df['cons'])

# 3. Save the DataFrame to a CSV file named "glassdoor_job_reviews_optimized.csv"
df.to_csv("glassdoor_job_reviews_optimized.csv", index=False)

Processing 'Pros'...


100%|██████████| 10000/10000 [00:21<00:00, 472.84it/s]


Processing 'Cons'...


100%|██████████| 10000/10000 [00:34<00:00, 292.19it/s]


In [45]:
df

,company_name,date_review,job title,location,overall rating,work life balance,culture_and_values,diversity inclusion,career_opportunities,compensation_and_benefits,...,headline,pros,cons,is_current_employee,employment type,seniority,cons_review_length,pros_review_length,pros_optimized,cons_optimized
0,AFH-Wealth-Management,2015-04-05,,NaN,2,4.0,3.0,NaN,2.0,3.0,...,"Young colleagues, poor micro management",Very friendly and welcoming to new staff. Easy...,"Poor salaries, poor training and communication.",True,Employee,None,6,10,staff,salary training communication
1,AFH-Wealth-Management,2015-12-11,Office Administrator,"Bromsgrove, England, England",2,3.0,1.0,NaN,2.0,1.0,...,"Excellent staff, poor salary","Friendly, helpful and hard-working colleagues",Poor salary which doesn't improve much with pr...,True,Employee,more than 1 year,19,5,colleague,salary progression incentive turnover staff sy...
2,AFH-Wealth-Management,2016-01-28,Office Administrator,"Bromsgrove, England, England",1,1.0,1.0,NaN,1.0,1.0,...,"Low salary, bad micromanagement",Easy to get the job even without experience in...,"Very low salary, poor working conditions, very...",True,Employee,less than 1 year,33,10,job experience finance,salary working condition training expectation ...
3,AFH-Wealth-Management,2016-04-16,,NaN,5,2.0,3.0,NaN,2.0,2.0,...,Over promised under delivered,Nice staff to work with,No career progression and salary is poor,True,Employee,None,7,5,staff,career progression salary
4,AFH-Wealth-Management,2016-04-23,Office Administrator,"Bromsgrove, England, England",1,2.0,1.0,NaN,2.0,1.0,...,client reporting admin,"Easy to get the job, Nice colleagues.","Abysmal pay, around minimum wage. No actual tr...",True,Employee,more than 1 year,41,7,job colleague,pay wage training job role incentive training ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Accor,2020-09-18,Meeting Planner,"Bedford, England, England",3,NaN,NaN,NaN,NaN,NaN,...,Average,Good team and management at work,Low pay for intensity of work,True,Employee,more than 3 years,6,6,team management work,pay intensity work
9996,Accor,2020-09-20,General Manager,"Birmingham, England, England",4,4.0,4.0,NaN,4.0,3.0,...,Good opportunities,Lots of training for those who want a career i...,Politics further up the company,False,Employee,more than 10 years,5,11,training career hospitality,politic
9997,Accor,2020-09-21,Customer Service,Sydney,5,NaN,NaN,NaN,NaN,NaN,...,Very Good,Excellent place to work in Aus,I had enjoyed my journey with them,False,Employee,None,7,6,aus,journey
9998,Accor,2020-09-23,Assistant Finance Manager,"London, England, England",2,2.0,1.0,NaN,2.0,4.0,...,Accor,Free meal so I don't have to cook,"very operational, not much excitement",True,Employee,more than 5 years,5,8,meal,excitement


In [52]:
print("Top 20 n-grams in cons_optimized:")
print(get_top_ngrams_optimized(df, 'cons_optimized').head(20))

print("\nTop 20 n-grams in pros_optimized:")
print(get_top_ngrams_optimized(df, 'pros_optimized').head(20))

Top 20 n-grams in cons_optimized:
normalized
hour long             652
balance life          382
life work             378
leader section        241
hard work             212
management poor       167
management senior     143
low pay               139
career progression    115
low salary            114
hour work              94
bad management         92
member staff           92
manager store          88
hour week              87
long shift             82
floor shop             82
management staff       81
hour shift             81
hour time              81
Name: count, dtype: int64

Top 20 n-grams in pros_optimized:
normalized
great people            487
flexible hour           272
balance life            270
life work               263
friendly staff          244
nice people             222
flexible working        187
great team              187
discount staff          183
benefit great           174
colleague friendly      173
great opportunity       144
colleague great         133

In [47]:
print("Top 20 n-grams in cons_optimized:")
top_20_cons = get_top_ngrams_optimized(df, 'cons_optimized', ngram_range=(3, 3))
print(top_20_cons.head(20))

print("\nTop 20 n-grams in pros_optimized:")
top_20_pros = get_top_ngrams_optimized(df, 'pros_optimized', ngram_range=(3, 3))
print(top_20_pros.head(20))

Top 20 n-grams in cons_optimized:
normalized
balance life work                 345
leader manager section             53
leader management section          28
management performance process     27
hour life work                     25
balance hour life                  20
contract hour                      19
colleague leader section           19
communication lack management      18
balance life project               18
hour week                          17
hour shift                         16
break hour shift                   15
management staff                   12
management member staff            11
lack life work                     11
leader section work                11
hour time week                     11
day hour shift                     11
area manager store                 11
Name: count, dtype: int64

Top 20 n-grams in pros_optimized:
normalized
balance life work                 243
discount sale sample               36
career opportunity progression     24
life peop

In [ ]:
print("Top 20 n-grams in cons_optimized:")
top_20_cons = get_ngrams_optimized(df, 'cons_optimized', ngram_range=(2, 3))
print(top_20_cons.head(20))

print("\nTop 20 n-grams in pros_optimized:")
top_20_pros = get_ngrams_optimized(df, 'pros_optimized', ngram_range=(2, 3))
print(top_20_pros.head(20))

Top 20 n-grams in cons_optimized:
normalized
hour long             672
balance life          382
life work             378
balance life work     342
leader section        241
hard work             212
management poor       183
low pay               144
management senior     143
low salary            118
career progression    115
hour week             100
hour work              94
manager store          93
bad management         92
member staff           92
hour shift             85
floor shop             82
long shift             82
management staff       81
Name: count, dtype: int64

Top 20 n-grams in pros_optimized:
normalized
great people            534
flexible hour           272
balance life            270
life work               263
friendly staff          244
balance life work       243
nice people             222
great team              195
flexible working        187
discount staff          183
benefit great           182
colleague friendly      173
great opportunity       144

In [ ]:
df = pd.read_csv("./glassdoor_job_reviews.csv")

In [69]:
print("Processing 'Pros'...")
df['pros_optimized'] = optimize_with_pipe(df['pros'])

print("Processing 'Cons'...")
df['cons_optimized'] = optimize_with_pipe(df['cons'])

df.to_csv("glassdoor_job_reviews_optimized.csv", index=False)

Processing 'Pros'...


100%|██████████| 805047/805047 [1:29:36<00:00, 149.74it/s]  


Processing 'Cons'...


100%|██████████| 805047/805047 [1:03:32<00:00, 211.17it/s] 


In [6]:
# Load the optimized reviews file into df
df = pd.read_csv("glassdoor_job_reviews_optimized.csv")

In [ ]:
def get_ngrams_optimized(df, column_name, ngram_range=(2, 2)):
    """
    Calculates the most frequent n-grams, normalizes word order 
    (canonicalizes phrases) and returns the Top N results.
    """
    # 1. Extract a larger list of phrases (e.g., 2000) to allow for grouping.
    # We do NOT use max_features=20 here, because we would lose variations we want to merge.
    vec = CountVectorizer(ngram_range=ngram_range, max_features=2000)
    matrix = vec.fit_transform(df[column_name].fillna(''))
    
    # 2. Count frequencies (using .A1 for flattening the matrix)
    counts_array = matrix.sum(axis=0).A1
    phrases = vec.get_feature_names_out()
    
    # 3. Create a temporary DataFrame for grouping
    df_freq = pd.DataFrame({'phrase': phrases, 'count': counts_array})
    
    # 4. NORMALIZATION: Sort words alphabetically within each phrase and remove duplicates
    # Using 'set()' helps to avoid cases like "good good pay" -> "good pay"
    df_freq['normalized'] = df_freq['phrase'].apply(lambda x: " ".join(sorted(list(set(x.split())))))
    
    # 5. GROUPING: Merge identical normalized phrases (e.g., "life balance" and "balance life")
    final_counts = df_freq.groupby('normalized')['count'].sum().sort_values(ascending=False)
    
    # Return only the Top N results
    return final_counts

In [ ]:
print("Top 20 n-grams in cons_optimized:")
cons = get_ngrams_optimized(df, 'cons_optimized', ngram_range=(2, 3))
print(cons.head(20))

print("\nTop 20 n-grams in pros_optimized:")
pros = get_ngrams_optimized(df, 'pros_optimized', ngram_range=(2, 3))
print(pros.head(20))

Top 20 n-grams in cons_optimized:
normalized
life work             45582
balance life          44851
balance life work     43019
hour long             41270
low pay               14351
                      ...  
employee time          1801
day week               1791
advancement career     1787
low wage               1785
leader team            1765
Name: count, Length: 100, dtype: int64

Top 20 n-grams in pros_optimized:
normalized
balance life         58128
life work            56239
balance life work    54665
great people         36755
benefit great        29939
                     ...  
new technology        2765
people team           2684
exposure great        2679
great perk            2628
schedule work         2613
Name: count, Length: 100, dtype: int64


In [ ]:
with open("top_300_ngrams_pros_cons.txt", "w", encoding="utf-8") as f:
    f.write("Top 300 n-grams in cons_optimized:\n")
    f.write(cons.head(300).to_string())
    f.write(f"\nSum of counts of top 300 cons n-grams: {cons.head(300).sum()}")

    f.write("\n\nTop 300 n-grams in pros_optimized:\n")
    f.write(pros.head(300).to_string())
    f.write(f"\nSum of counts of top 300 pros n-grams: {pros.head(300).sum()}")

In [45]:
def get_assigned_themes(
    row,
    text_col: str,
    theme_map: dict,
):
    """
    Iterates over the n-grams of the specified column (text_col) in a row.
    Returns all unique themes whose n-grams appear in that row (according to the theme_map).
    Returns a sorted list (or pd.NA if none found). Can be used with .apply(..., axis=1).
    theme_map: dict {theme: [ngram, ngram, ...]}
    """
    text = row[text_col]
    if pd.isna(text) or not text:
        return pd.NA
    tokens = text.split()
    found_themes = set()
    ngram_set = set()
    max_ngram_size = max(len(ngrams.split()) for themes in theme_map.values() for ngrams in themes)
    min_ngram_size = min(len(ngrams.split()) for themes in theme_map.values() for ngrams in themes)
    # Generate all possible n-grams in the row text (according to min-max theme_map)
    for n in range(min_ngram_size, max_ngram_size + 1):
        for i in range(len(tokens) - n + 1):
            ng = " ".join(sorted(set(tokens[i:i + n])))
            ngram_set.add(ng)
    # Check which theme_map n-grams appear
    for theme, ngrams in theme_map.items():
        for ng in ngrams:
            norm_ng = " ".join(sorted(set(ng.split())))
            if norm_ng in ngram_set:
                found_themes.add(theme)
    if not found_themes:
        return pd.NA

    return sorted(found_themes)

In [47]:
with open('review_theme_maps.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

theme_map_pros = data['theme_map_pros']
theme_map_cons = data['theme_map_cons']

df['assigned_pros_themes'] = df.apply(get_assigned_themes, axis=1, text_col='pros_optimized', theme_map=theme_map_pros)
df['assigned_cons_themes'] = df.apply(get_assigned_themes, axis=1, text_col='cons_optimized', theme_map=theme_map_cons)

df.to_csv("glassdoor_job_reviews_optimized.csv", index=False)
df

,company_name,date_review,job title,location,overall rating,work life balance,culture_and_values,diversity inclusion,career_opportunities,compensation_and_benefits,...,cons,is_current_employee,employment type,seniority,cons_review_length,pros_review_length,pros_optimized,cons_optimized,assigned_pros_themes,assigned_cons_themes
0,AFH-Wealth-Management,2015-04-05,,NaN,2,4.0,3.0,NaN,2.0,3.0,...,"Poor salaries, poor training and communication.",True,Employee,NaN,6,10,friendly new staff easy ethic,poor salary poor training communication,<NA>,"[Salary & Compensation, Workplace Culture & En..."
1,AFH-Wealth-Management,2015-12-11,Office Administrator,"Bromsgrove, England, England",2,3.0,1.0,NaN,2.0,1.0,...,Poor salary which doesn't improve much with pr...,True,Employee,more than 1 year,19,5,friendly helpful colleague,poor salary progression incentive high turnove...,[People & Culture],"[Salary & Compensation, Workplace Culture & En..."
2,AFH-Wealth-Management,2016-01-28,Office Administrator,"Bromsgrove, England, England",1,1.0,1.0,NaN,1.0,1.0,...,"Very low salary, poor working conditions, very...",True,Employee,less than 1 year,33,10,easy job experience finance,low salary poor working condition little train...,"[Career & Development, Work Environment & Job ...","[Management & Leadership, Salary & Compensatio..."
3,AFH-Wealth-Management,2016-04-16,,NaN,5,2.0,3.0,NaN,2.0,2.0,...,No career progression and salary is poor,True,Employee,NaN,7,5,nice staff,career progression salary poor,[People & Culture],"[Career & Growth, Salary & Compensation]"
4,AFH-Wealth-Management,2016-04-23,Office Administrator,"Bromsgrove, England, England",1,2.0,1.0,NaN,2.0,1.0,...,"Abysmal pay, around minimum wage. No actual tr...",True,Employee,more than 1 year,41,7,easy job nice colleague,abysmal pay minimum wage actual training job r...,"[People & Culture, Work Environment & Job Sati...",[Salary & Compensation]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805042,the-LEGO-Group,2021-06-02,Marketing Manager,"München, Bavaria, Bavaria",5,4.0,5.0,4.0,4.0,4.0,...,Not very easy to transfer to other locations,True,Employee,more than 5 years,8,10,great value awesome product smart colleague st...,easy location,[People & Culture],<NA>
805043,the-LEGO-Group,2021-06-03,Sales Associate,"London, England, England",3,NaN,NaN,NaN,NaN,NaN,...,micro managing is a hassle\r\ncan become menta...,True,Employee,less than 1 year,9,5,staff discount nice,micro hassle,[Benefits & Perks],<NA>
805044,the-LEGO-Group,2021-06-03,Strategist,NaN,4,5.0,5.0,5.0,3.0,5.0,...,you can spend 6-10 years without any promotion...,True,Employee,NaN,13,7,brand people,year promotion rule,<NA>,<NA>
805045,the-LEGO-Group,2021-06-04,Customer Service Representative,NaN,5,NaN,NaN,NaN,NaN,NaN,...,"Working every other weekend, busy seasons can ...",True,Employee,less than 1 year,9,7,wage hour resource,weekend busy season stressful,<NA>,[Work-Life Balance & Working Hours]


In [52]:
def theme_count_df(assigned_themes_series, is_pro):
    theme_counts = Counter()

    for themes in assigned_themes_series:
        if isinstance(themes, list):
            theme_counts.update(themes)

    return pd.DataFrame({
        "theme": list(theme_counts.keys()),
        "count": [theme_counts[theme] for theme in theme_counts],
        "is_pro": [is_pro for _ in theme_counts]
    })

pros_df = theme_count_df(df['assigned_pros_themes'], True)
cons_df = theme_count_df(df['assigned_cons_themes'], False)

# Concatenate pros and cons (sujungti i viena df)
themes_df = pd.concat([pros_df, cons_df], ignore_index=True)

# Sort for easier inspection/view
themes_df = themes_df.sort_values(["is_pro", "count"], ascending=[False, False], ignore_index=True)

themes_df.to_csv('themes_statistic.csv', index=False, encoding='utf-8')

In [53]:
themes_df

,theme,count,is_pro
0,People & Culture,189998,True
1,Work Environment & Job Satisfaction,135911,True
2,Work-Life Balance & Flexibility,115488,True
3,Career & Development,95836,True
4,Benefits & Perks,91050,True
5,Compensation,32025,True
6,Work-Life Balance & Working Hours,119603,False
7,Workplace Culture & Environment,81054,False
8,Management & Leadership,71642,False
9,Salary & Compensation,62873,False


In [ ]:
def get_ngram_theme(ngram, theme_map):
    for theme, ngrams in theme_map.items():
        if ngram in ngrams:
            return theme
    return pd.NA

def get_ngrams_with_themes(top_ngrams, is_pro, theme_map):
    rows = []
    for ngram, count in top_ngrams.items():
        theme = get_ngram_theme(ngram, theme_map)
        if not pd.isna(theme):
            rows.append({
                "theme": theme,
                "ngram": ngram,
                "count": count,
                "is_pro": is_pro,
            })
    return rows

top_cons = cons.head(100)
top_pros = pros.head(100)

pros_cons_list = []
pros_cons_list.extend(get_ngrams_with_themes(top_cons, False, theme_map_cons))
pros_cons_list.extend(get_ngrams_with_themes(top_pros, True, theme_map_pros))

pros_cons_df = pd.DataFrame(pros_cons_list)
pros_cons_df.to_csv("top_100_ngrams_pros_cons.csv", index=False, encoding="utf-8")

In [54]:
# Extract early leavers: former employees with less than 1 year and less than 3 years of seniority
early_leaves_1_df = df[
    (df['is_current_employee'] == False) &
    (df['seniority'] == "less than 1 year")
]

early_leaves_3_df = df[
    (df['is_current_employee'] == False) &
    (df['seniority'] == "more than 3 years")
]

# Extract long stays: current employees and those with more than 5/10 years of seniority
long_stays_5_df = df[
    (df['is_current_employee'] == True) &
    (df['seniority'] == "more than 5 years")
]

long_stays_10_df = df[
    (df['is_current_employee'] == True) &
    (df['seniority'] == "more than 10 years")
]

# Optionally display the head of each DataFrame
display(early_leaves_1_df.head())
display(early_leaves_3_df.head())
display(long_stays_5_df.head())
display(long_stays_10_df.head())

,company_name,date_review,job title,location,overall rating,work life balance,culture_and_values,diversity inclusion,career_opportunities,compensation_and_benefits,...,cons,is_current_employee,employment type,seniority,cons_review_length,pros_review_length,pros_optimized,cons_optimized,assigned_pros_themes,assigned_cons_themes
34,AFH-Wealth-Management,2019-05-02,Anonymous Employee,NaN,1,1.0,1.0,NaN,1.0,2.0,...,Training is rush not a nice environment too ma...,False,Employee,less than 1 year,10,1,NaN,training rush nice environment clique,<NA>,<NA>
35,AFH-Wealth-Management,2019-07-28,Financial Advisor Trainee,"Birmingham, England, England",1,2.0,1.0,NaN,1.0,1.0,...,Lack of support to develop. The HR team is a s...,False,Employee,less than 1 year,11,17,package mis true,lack support hr team shamble,<NA>,[Workplace Culture & Environment]
36,AFH-Wealth-Management,2019-07-30,Protection Consultant,"Nottingham, England, England",1,3.0,1.0,NaN,1.0,1.0,...,"i worked at their AFH Insure division, give yo...",False,Employee,less than 1 year,93,9,hour ok colleague line manager nice,afh insure division lead impossible existent d...,[Work Environment & Job Satisfaction],<NA>
49,AFH-Wealth-Management,2020-06-08,Anonymous Employee,"Bromsgrove, England, England",1,3.0,1.0,NaN,1.0,1.0,...,Supervisor made me very uneasy from the start ...,False,Employee,less than 1 year,46,25,hour people actual boss supervisor supportive ...,supervisor uneasy start rest team people perso...,<NA>,<NA>
52,AFH-Wealth-Management,2021-02-07,IFA Administrator,"Bromsgrove, England, England",4,3.0,3.0,4.0,4.0,4.0,...,Management can be clicky at times,False,Employee,less than 1 year,6,7,nice environment love people stressful,management clicky time,[People & Culture],<NA>


,company_name,date_review,job title,location,overall rating,work life balance,culture_and_values,diversity inclusion,career_opportunities,compensation_and_benefits,...,cons,is_current_employee,employment type,seniority,cons_review_length,pros_review_length,pros_optimized,cons_optimized,assigned_pros_themes,assigned_cons_themes
9,AFH-Wealth-Management,2017-02-21,Technician,"Santa Ana, CA",1,1.0,1.0,NaN,1.0,3.0,...,Was let go from the company just before Christ...,False,Employee,more than 3 years,29,5,life time friend,christmas role effective bad management year c...,<NA>,[Management & Leadership]
26,AFH-Wealth-Management,2018-07-05,Anonymous Employee,NaN,1,3.0,1.0,NaN,2.0,1.0,...,The talented and friendly people are woefully ...,False,Employee,more than 3 years,43,15,talented friendly people real opportunity,talented friendly people management self short...,[People & Culture],"[Management & Leadership, Salary & Compensation]"
33,AFH-Wealth-Management,2019-04-11,Anonymous Employee,NaN,1,2.0,1.0,NaN,2.0,1.0,...,The environment is extrememely biased in that ...,False,Employee,more than 3 years,205,61,pro employment afh wealth management apparent ...,environment biased one time rewarding enjoyabl...,"[Benefits & Perks, Compensation]",[Workplace Culture & Environment]
40,AFH-Wealth-Management,2019-11-08,Compliance,NaN,5,4.0,5.0,NaN,3.0,4.0,...,Progression can be slow in some departments,False,Employee,more than 3 years,7,8,friendly staff knowledgable individual trainin...,progression slow department,[People & Culture],[Career & Growth]
50,AFH-Wealth-Management,2020-10-01,Office Administrator,"Bromsgrove, England, England",2,1.0,3.0,1.0,1.0,2.0,...,"Poor pay, huge gap for pay between senior mana...",False,Employee,more than 3 years,27,8,great people excellent christmas summer party,poor pay huge gap pay senior management manage...,[People & Culture],"[Management & Leadership, Salary & Compensation]"


,company_name,date_review,job title,location,overall rating,work life balance,culture_and_values,diversity inclusion,career_opportunities,compensation_and_benefits,...,cons,is_current_employee,employment type,seniority,cons_review_length,pros_review_length,pros_optimized,cons_optimized,assigned_pros_themes,assigned_cons_themes
7,AFH-Wealth-Management,2016-09-25,Anonymous Employee,"Century City, CA",5,5.0,5.0,NaN,5.0,4.0,...,Wouldn't necessarily say there are any cons to...,True,Employee,more than 5 years,36,22,people great culture thinking development oppo...,con website negative review committed loyalty,"[Career & Development, People & Culture]",<NA>
70,AJ-Bell,2016-04-26,Property Administrator,"Manchester, England, England",4,4.0,5.0,NaN,3.0,3.0,...,Underpaid therefore experienced staff leave ma...,True,Employee,more than 5 years,22,23,perk buy holiday holiday allowance increase ye...,underpaid experienced staff role wage people i...,<NA>,<NA>
85,AJ-Bell,2017-04-03,Anonymous Employee,"Manchester, England, England",5,4.0,5.0,NaN,5.0,3.0,...,"Current building is very shabby, but we're mov...",True,Employee,more than 5 years,49,71,recruitment area great person skill great team...,current building shabby opinion decision peopl...,"[Career & Development, People & Culture]",<NA>
97,AJ-Bell,2018-10-25,Anonymous Employee,"London, England, England",4,4.0,4.0,NaN,5.0,3.0,...,Very different cultures between Manchester & L...,True,Employee,more than 5 years,55,23,plenty opportunity right people afraid money m...,different culture manchester london office ben...,[Career & Development],<NA>
117,AJ-Bell,2020-06-09,Business Development,"Manchester, England, England",5,5.0,5.0,NaN,5.0,5.0,...,Due to location parking can be difficult for t...,True,Employee,more than 5 years,21,28,great office working environment site facility...,location parking difficult space public transp...,[Work Environment & Job Satisfaction],<NA>


,company_name,date_review,job title,location,overall rating,work life balance,culture_and_values,diversity inclusion,career_opportunities,compensation_and_benefits,...,cons,is_current_employee,employment type,seniority,cons_review_length,pros_review_length,pros_optimized,cons_optimized,assigned_pros_themes,assigned_cons_themes
87,AJ-Bell,2017-05-05,Anonymous Employee,"Manchester, England, England",4,4.0,4.0,NaN,4.0,4.0,...,Nature of business doesn't allow true flexible...,True,Employee,more than 10 years,8,15,opportunity support professional qualification...,nature business true flexible working,"[People & Culture, Work Environment & Job Sati...",<NA>
179,ALDI,2013-02-09,Store Manager,"Neston, North West England, England, England",3,2.0,2.0,NaN,2.0,4.0,...,"Long and often unpredictable hours, sometimes ...",True,Employee,more than 10 years,42,9,great salary sector challenging dull,long unpredictable hour high expectation total...,[Compensation],"[Management & Leadership, Workplace Culture & ..."
195,ALDI,2014-09-19,Store Manager,"Leeds, England, England",5,5.0,5.0,NaN,5.0,5.0,...,Having to work to speed great pay and being ab...,True,Employee,more than 10 years,43,55,love money speed day challenge problem job tea...,great pay able way people fast pace,<NA>,<NA>
210,ALDI,2015-05-09,Store Manager,"Penzance, England, England",1,1.0,3.0,NaN,1.0,1.0,...,always second guessing the company,True,Employee,more than 10 years,5,7,money comment,NaN,<NA>,<NA>
212,ALDI,2015-05-23,Store Assistant,"Huddersfield, England, England",2,1.0,3.0,NaN,1.0,1.0,...,The money isn't as good as you think because u...,True,Employee,more than 10 years,35,219,mild feeling accomplishment target work consta...,money favourite employee overtime high rate pa...,[Work Environment & Job Satisfaction],[Salary & Compensation]


In [55]:
def create_theme_df_from_column(theme_df, seniority, is_pro):
    theme_count = Counter()

    for themes in theme_df:
        if isinstance(themes, list):
            theme_count.update(themes)

    return pd.DataFrame({
        "theme": list(theme_count.keys()),
        "count": [theme_count[theme] for theme in theme_count],
        "is_pro": [is_pro for _ in theme_count],
        "seniority": seniority
    })

early_leaves_1_cons_df = create_theme_df_from_column(early_leaves_1_df['assigned_cons_themes'], "less than 1 year", False)
early_leaves_3_cons_df = create_theme_df_from_column(early_leaves_3_df['assigned_cons_themes'], "more than 3 years", False)
long_stays_5_pros_df = create_theme_df_from_column(long_stays_5_df['assigned_pros_themes'], "more than 5 years", True)
long_stays_10_pros_df = create_theme_df_from_column(long_stays_10_df['assigned_pros_themes'], "more than 10 years", True)

themes_df = pd.concat([
    early_leaves_1_cons_df, 
    early_leaves_3_cons_df, 
    long_stays_5_pros_df, 
    long_stays_10_pros_df
], 
ignore_index=True)

themes_df.to_csv('early_leaves_long_stays_themes.csv', index=False, encoding='utf-8')